In [4]:
import os
from os import makedirs
import pandas as pd

from tqdm         import tqdm
from datasets     import load_dataset
from transformers import AutoTokenizer
from mistralai    import Mistral


# General info
api_key = 'USE_YOUR_OWN_KEY'
client  = Mistral(api_key=api_key)
models  = {
    #'mistral-nemo': 'open-mistral-nemo',
    'mistral-7b'  : {
        'api-id'  : 'open-mistral-7b', 
        'hf-id'   : 'mistralai/Mistral-7B-Instruct-v0.3',
        'context' : 32000
    },
}

# prepare data
dataset = load_dataset("arch-be/brabant-xvii", name="doc_by_doc")
train   = dataset['train'].to_pandas()
valid   = dataset['valid'].to_pandas()
test    = dataset['test'].to_pandas()
dataset = pd.concat([train, test, valid]).set_index(['project', 'file_id'])
# start with the longest texts (fail fast)
dataset.sort_values(by='text', key=lambda x: x.str.len(), ascending=False, inplace=True)

with open('prompt.txt', encoding='utf8') as f:
    prompt = f.read()

# run the inference pipeline
for model, data in models.items():
    makedirs(f'generated_texts/{model}/', exist_ok=True)
    tokenizer = AutoTokenizer.from_pretrained(data['hf-id'])
    #
    for id, text in tqdm(dataset['text'].items()):
        project,file_id = id
        #print(f'generated_texts/{model}/{project}_{file_id}')
        fname = f'generated_texts/{model}/{project}_{file_id}'
        if os.path.exists(fname):
            continue
        else:
            with open(fname, 'w', encoding='utf8') as f:
                messages = [
                        {'role': 'system', 'content': prompt},
                        {'role': 'user',   'content': text},
                    ]
                tokens   = tokenizer.encode(str(messages), truncation=False)
                if len(tokens) >= data['context']:
                    print('## fail: the prompt is too long for this model', file=f)
                else:
                    response = client.chat.complete(
                        model=data['api-id'],
                        response_format={'type': 'json_object'},
                        messages = messages,
                    )
                    print(response, file=f)

419it [06:41,  1.04it/s]
